# TP 7 : Introduction à TensorFlow et Keras pour l'Apprentissage Profond

### Identification de l'étudiant

In [ ]:
# Nom : 
# Prénom : 
# Numéro d'étudiant : 

### Modalités de rendu

1. Chaque étudiant doit rendre un travail individuel.
2. Renommez ce fichier selon la convention : `TP7_Nom_Prenom.ipynb`.
3. Le rendu s'effectuera via le lien de dépôt communiqué par votre enseignant.
4. Assurez-vous que votre code s'exécute sans erreur (Menu : Kernel > Restart & Run All).

### Objectifs de la séance

L'objectif de cette séance est d'opérer la transition entre l'apprentissage statistique classique (Scikit-Learn) et les réseaux de neurones profonds (TensorFlow / Keras). À l'issue de ce TP, vous serez capables de :
- Maîtriser les paradigmes de modélisation Keras (API Séquentielle vs Fonctionnelle).
- Traiter un jeu de données réel en prévenant les fuites de données (Data Leakage).
- Analyser et contrer le phénomène de surapprentissage (Overfitting) via la régularisation mathématique (L2, Dropout).
- Optimiser l'entraînement dynamiquement grâce aux Callbacks.

### Documentation utile
- [Keras Sequential API](https://keras.io/guides/sequential_model/)
- [Keras Functional API](https://keras.io/guides/functional_api/)
- [Keras Callbacks](https://keras.io/api/callbacks/)

---

## Partie 1 : Concepts Fondamentaux et Pont d'API (MNIST)

Avant de manipuler des données complexes, nous allons comparer la philosophie de `scikit-learn` et de `Keras` sur le célèbre jeu de données MNIST (reconnaissance de chiffres manuscrits). 

![MNIST](https://upload.wikimedia.org/wikipedia/commons/2/27/MnistExamples.png)

### 1.1 Importation et Préparation
Importez les librairies standards et chargez le jeu de données MNIST. N'oubliez pas que les réseaux de neurones optimisés par descente de gradient exigent des données normalisées.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Importation directe de Keras
import keras
from keras.models import Sequential, Model
from keras.layers import Input, Dense, Dropout, BatchNormalization
from keras.regularizers import l2
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

sns.set_theme(style="whitegrid")

In [ ]:
# Chargement de MNIST
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.mnist.load_data()

# Les images font 28x28 pixels. Nous devons les aplatir (flatten) en vecteurs de dimension 784.
X_train_flat = X_train_full.reshape(-1, ...).astype('float32')
X_test_flat = X_test.reshape(-1, ...).astype('float32')

# Normalisation : Les pixels vont de 0 à 255. Divisez par 255 pour ramener les valeurs dans [0, 1].
X_train_norm = X_train_flat / ...
X_test_norm = X_test_flat / ...

print(f"Dimensions d'entraînement : {X_train_norm.shape}")

### 1.2 La "Boîte Noire" Scikit-Learn
Dans `scikit-learn`, la topologie du réseau, la boucle de gradient et la gestion des tenseurs sont encapsulées dans la classe `MLPClassifier`. L'utilisateur ne voit qu'une interface abstraite.

In [ ]:
# Instanciez un perceptron multicouche avec une couche cachée de 64 neurones.
sk_model = MLPClassifier(hidden_layer_sizes=(...), max_iter=10, random_state=42)

# Entraînez le modèle (fit) et prédisez sur l'échantillon de test
sk_model....(X_train_norm, y_train_full)
sk_preds = sk_model....(X_test_norm)

print(f"Précision Scikit-Learn : {accuracy_score(y_test, sk_preds):.4f}")

### 1.3 L'API Sequential de Keras (L'approche "Lego")
Keras nous oblige à définir mathématiquement les transformations. L'API séquentielle convient parfaitement pour empiler des couches linéaires (chaque couche a exactement un tenseur d'entrée et un tenseur de sortie).

**Rappel :** La couche de sortie pour un problème de classification multi-classes (10 chiffres) nécessite 10 neurones et une fonction d'activation `softmax` (qui transforme les logits en probabilités).

In [ ]:
model_seq = Sequential([
    Dense(64, activation='...', input_shape=(784,)),
    Dense(..., activation='...')
])

# Contrairement à sklearn, nous devons "compiler" le modèle en spécifiant l'optimiseur et la métrique de perte.
model_seq.compile(optimizer='adam', 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])

# Entraînement (Keras gère les epochs et la taille des mini-lots)
model_seq.fit(X_train_norm, y_train_full, epochs=2, batch_size=128, validation_split=0.1)

### 1.4 L'API Fonctionnelle de Keras (Composition Mathématique)

**Pourquoi utiliser l'API fonctionnelle ?**
Imaginons un modèle estimant le prix d'une maison. Il reçoit une *image* (plan) et des *données tabulaires* (nombre de pièces). Un modèle séquentiel ne peut pas accepter deux entrées distinctes. 
L'API fonctionnelle traite chaque couche comme une fonction mathématique applicable à un tenseur : $y = f(h(x))$. Elle permet de créer des graphes complexes (branchements, concaténations, connexions résiduelles).

In [ ]:
# Définition du tenseur d'entrée x
inputs = Input(shape=(784,))

# Application de la fonction g (couche cachée) sur x : h = g(x)
h1 = Dense(64, activation='relu')(inputs)

# Application de la fonction f (couche de sortie) sur h : y = f(h)
outputs = Dense(10, activation='softmax')(...) 

# Création du graphe global
model_func = Model(inputs=..., outputs=...)

# Compile et fit
model_func.compile(...)
model_func.fit(...)

# Affichez le résumé de l'architecture via la méthod `summary`. Observez le nombre total de paramètres (poids + biais).
model_func....()

---

## Partie 2 : Application au jeu de données Kepler

Le jeu de données `cumulative.csv` contient les relevés du télescope spatial Kepler. Le but est de prédire si une observation est une véritable exoplanète ou un artefact (faux positif) à partir de ses caractéristiques physiques (profondeur du transit, température stellaire, etc.).

**La méthode des transits :** Lorsqu'une planète passe devant son étoile, elle bloque une fraction de la lumière. La courbe de lumière (comme celle de WASP-96b capturée par le télescope James Webb ci-dessous) forme un \"U\". La profondeur de ce transit (`koi_depth`) et sa durée (`koi_duration`) sont des variables physiques clés de notre jeu de données.

![Transit Exoplanète](https://upload.wikimedia.org/wikipedia/commons/6/60/Exoplanet_WASP-96_b_%28NIRISS_Transit_Light_Curve%29_%28weic2206b%29.jpeg?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=original)

### 2.1 Chargement et Fuite de Données (Data Leakage)

Chargez le fichier et affichez ses dimensions.

In [ ]:
df_kepler = pd.read_csv('cumulative.csv')
display(df_kepler.head(3))
print("Dimensions initiales :", df_kepler.shape)

**Attention aux fuites de données !** 
La variable cible est `koi_disposition`. Les valeurs `CANDIDATE` représentent une incertitude scientifique (vérité terrain inconnue). Si nous entraînons le modèle dessus, nous ne faisons qu'apprendre l'incertitude des chercheurs.
De plus, ce dataset contient des indicateurs de faux positifs (`koi_fpflag_*`). Si nous laissons ces variables, le réseau neuronal va trivialement apprendre une règle "Si flag == 1 alors Faux Positif" sans analyser la physique. Il faut les retirer.

In [ ]:
# 1. Ne conservez que les lignes où 'koi_disposition' est 'CONFIRMED' ou 'FALSE POSITIVE'
df_kepler = df_kepler[df_kepler['koi_disposition'].isin(['...', '...'])]

# 2. Encodage binaire de la cible : CONFIRMED = 1, FALSE POSITIVE = 0
df_kepler['target'] = df_kepler['koi_disposition'].map({'...': 1, '...': 0})

# 3. Sélection stricte des 15 variables physiques pertinentes (exclusion des fuites et des erreurs de mesure)
features_physiques = [
    'koi_period', 'koi_time0bk', 'koi_impact', 'koi_duration', 'koi_depth', 
    'koi_prad', 'koi_teq', 'koi_insol', 'koi_model_snr', 'koi_steff', 
    'koi_slogg', 'koi_srad', 'ra', 'dec', 'koi_kepmag'
]

df_physique = df_kepler[features_physiques + ['target']].copy()

# 4. Supprimez les lignes contenant des valeurs manquantes (NaN)
df_clean = df_physique....()

print("Dimensions après nettoyage pour la modélisation physique :", df_clean.shape)

### 2.2 Prétraitement et Standardisation
Séparez les données (Train/Test à 80/20) puis appliquez une standardisation.

**Rappel Mathématique :** La fonction de perte d'un réseau neuronal navigue dans un espace de très grande dimension. Si les variables ne sont pas à la même échelle (ex: période en jours vs température en milliers de Kelvins), les gradients seront disproportionnés et l'optimiseur oscillera de manière instable.

In [ ]:
X = df_clean.drop(columns=['target']).values
y = df_clean['target'].values

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(..., ..., test_size=0.2, random_state=42)

# Standardisation (Rappel : fit_transform sur train, transform sur test)
scaler = StandardScaler()
X_train_scaled = scaler....(...)
X_test_scaled = scaler....(...)

input_dim = X_train_scaled.shape[1]

### 2.3 Construction du Modèle : Séquentiel vs Fonctionnel

Nous allons construire un réseau de neurones profond (Deep Neural Network) pour traiter nos 15 caractéristiques physiques. L'architecture retenue est la suivante : 
**Entrée (15) $\rightarrow$ Dense(128) $\rightarrow$ Dense(64) $\rightarrow$ Dense(32) $\rightarrow$ Dense(1)** (Sortie binaire).

Afin de maîtriser les deux paradigmes de Keras, vous devez implémenter cette architecture exacte deux fois : une fois via l'API Séquentielle, et une fois via l'API Fonctionnelle. Vous vérifierez ensuite que le nombre de paramètres à optimiser est strictement identique.

In [ ]:
# 1. Implémentation via l'API Séquentielle
model_seq = Sequential([
    Dense(128, activation='...', input_shape=(input_dim,)),
    Dense(..., activation='...'),
    Dense(..., activation='...'),
    Dense(1, activation='...')  # Activation pour probabilité binaire. Références: https://keras.io/api/layers/activations/
])

# 2. Implémentation via l'API Fonctionnelle (Composition mathématique)
inputs = Input(shape=(input_dim,))
h1 = Dense(128, activation='relu')(inputs)
h2 = Dense(..., activation='...')(h1)
h3 = Dense(..., activation='...')(h2)
outputs = Dense(1, activation='...')(h3)

model_func = Model(inputs=..., outputs=...)


Utilisez la méthode `.summary()` pour inspecter les deux modèles. Comparez le nombre total de paramètres (poids et biais).

In [ ]:
print("--- Modèle Séquentiel ---")
model_seq....()

print("\n--- Modèle Fonctionnel ---")
model_func....()

# Vérification programmatique
assert model_seq.count_params() == model_func.count_params(), "Les topologies diffèrent !"

### 2.4 Entraînement du Modèle

Nous utiliserons le modèle séquentiel pour la suite. Compilez-le avec l'optimiseur Adam et la fonction de coût adaptée à la classification binaire. L'entraînement s'effectuera sur **70 itérations (epochs)**.

In [ ]:
model_seq.compile(optimizer='adam', loss='...', metrics=['accuracy'])

print("Entraînement en cours...")
history_base = model_seq.fit(X_train_scaled, y_train, 
                             epochs=70, batch_size=128, 
                             validation_split=0.2)

Utilisez la fonction ci-dessous pour tracer les courbes d'apprentissage de votre modèle.

In [ ]:
# Affiche les clés générées dynamiquement par Keras
print(history_base.history.keys())


def plot_history(history, title="Courbes d'apprentissage"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.plot(history.history['...'], label='Train Loss')
    ax1.plot(history.history['...'], label='Validation Loss')
    ax1.set_title(f"{title} - Loss")
    ax1.legend()
    
    ax2.plot(history.history['...'], label='Train Acc')
    ax2.plot(history.history['...'], label='Validation Acc')
    ax2.set_title(f"{title} - Accuracy")
    ax2.legend()
    
    plt.show()

plot_history(history_base, "Modèle de Base (128->64->32->1)")

**Analyse :** Double-cliquez ici pour décrire le comportement des deux courbes de Loss (Train vs Validation). À partir de quelle itération (epoch) environ observez-vous un décrochage ? Quel est le nom mathématique de ce phénomène en apprentissage statistique ?

---

## Partie 3 : Régularisation, Callbacks et Sauvegarde

L'entraînement de réseaux profonds est coûteux en temps de calcul. Comme observé précédemment, le modèle peut atteindre son point optimal (minimum de la `val_loss`) aux alentours d'une certaine epoch, puis se dégrader sur les itérations suivantes.

Il est inefficace de devoir deviner le nombre exact d'epochs à l'avance. Nous faisons donc :
1. **Régulariser** le modèle pour retarder et atténuer le phénomène observé en Partie 2.
2. Surveiller l'entraînement dynamiquement via des **Callbacks**.
3. **Sauvegarder physiquement** sur le disque dur l'état exact du modèle lorsqu'il atteint son optimum de validation, afin de ne jamais perdre les meilleurs poids calculés.

### 3.1 Évaluation Isolée des Techniques de Régularisation

Pour comprendre mathématiquement l'impact de la régularisation, nous n'allons pas toutes les mélanger. Nous allons tester séparément trois approches sur notre architecture de base (128 -> 64 -> 32 -> 1) et observer comment elles modifient la surface de la fonction de coût.

**1. Pénalité L2 (Ridge)**
Elle ajoute le terme $\lambda \|\mathbf{W}\|_2^2$ à la Loss. Complétez le modèle ci-dessous avec `kernel_regularizer=l2(0.01)`.

In [ ]:
from keras.regularizers import l2

model_l2 = Sequential([
    Dense(128, activation='relu', kernel_regularizer=..., input_shape=(input_dim,)),
    Dense(64, activation='relu', kernel_regularizer=...),
    Dense(32, activation='relu', kernel_regularizer=...),
    Dense(1, activation='sigmoid')
])

model_l2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_l2 = model_l2.fit(X_train_scaled, y_train, epochs=70, batch_size=128, validation_split=0.2)

plot_history(history_l2, "1. Effet de la Pénalité L2")

**2. Dropout**
Cette méthode stochastique désactive aléatoirement une fraction $p$ des neurones à chaque itération. Insérez des couches `Dropout(0.3)` après chaque couche Dense cachée.

In [ ]:
from keras.layers import Dropout

model_dropout = Sequential([
    Dense(128, activation='relu', input_shape=(input_dim,)),
    Dropout(...),
    Dense(64, activation='relu'),
    Dropout(...),
    Dense(32, activation='relu'),
    Dropout(...),
    Dense(1, activation='sigmoid')
])

model_dropout.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_dropout = model_dropout.fit(X_train_scaled, y_train, epochs=70, batch_size=128, validation_split=0.2)

plot_history(history_dropout, "2. Effet du Dropout")

**3. Batch Normalization**
Elle standardise les activations par mini-lot. Bien qu'elle soit avant tout une technique d'accélération de convergence (réduction du décalage de covariance interne), elle agit également comme un léger régularisateur.
Insérez des couches `BatchNormalization()` après chaque couche cachée.

In [ ]:
from keras.layers import BatchNormalization

model_bn = Sequential([
    Dense(128, activation='relu', input_shape=(input_dim,)),
    BatchNormalization(),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dense(1, activation='sigmoid')
])

model_bn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_bn = model_bn.fit(X_train_scaled, y_train, epochs=70, batch_size=128, validation_split=0.2)

plot_history(history_bn, "3. Effet de la Batch Normalization")

**Analyse comparative :** Double-cliquez ici. Quelle méthode semble la plus efficace pour empêcher la courbe de validation de diverger sur ce jeu de données spécifique ? Laquelle converge le plus vite ?

### 3.2 Optimisation Dynamique et Sauvegarde Automatique (Callbacks)

Nous allons utiliser trois Callbacks conjointement :
- `ReduceLROnPlateau` : Réduit le pas d'apprentissage si la progression stagne.
- `EarlyStopping` : Arrête l'entraînement si la `val_loss` remonte de manière persistante.
- `ModelCheckpoint` : Sauvegarde automatiquement le modèle (`.keras`) à chaque fois qu'un nouveau record de `val_loss` est atteint.

In [ ]:
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# 1. Callback de réduction du pas d'apprentissage
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, verbose=1)

# 2. Callback d'arrêt prématuré
early_stop = EarlyStopping(monitor='...', patience=10, verbose=1)

# 3. Callback de sauvegarde (Sauvegarde uniquement le meilleur modèle)
checkpoint = ModelCheckpoint(filepath='meilleur_modele_kepler.keras', 
                             monitor='...', 
                             save_best_only=..., 
                             verbose=1)

print("Entraînement avec Callbacks en cours...")
# Nous pouvons fixer un nombre d'epochs très élevé (ex: 150), les callbacks gèreront l'arrêt.
history_reg = model_bn.fit(X_train_scaled, y_train, 
                            epochs=150, batch_size=128, 
                            validation_split=0.2, 
                            callbacks=[..., ..., ...], 
                            verbose=1)

plot_history(history_reg, "Modèle Régularisé avec Callbacks")

### 3.3 Chargement et Évaluation du Meilleur Modèle
Grâce au `ModelCheckpoint`, l'état optimal du réseau (avant qu'il ne se dégrade) a été figé sur votre disque. Si vous redémarrez ce notebook demain, vous n'aurez pas besoin de relancer l'entraînement.

Chargez ce modèle sauvegardé et évaluez-le sur le jeu de test inédit.

In [ ]:
from keras.models import load_model

# Restauration du modèle optimal
modele_optimal = load_model('...')

# Évaluation finale sur X_test_scaled et y_test
test_loss, test_acc = modele_optimal.evaluate(..., ...)
print(f"Précision optimale sur le jeu de test : {test_acc:.4f}")

## Partie 4 : Conception Libre (Projet d'Autonomie)

Vous êtes désormais l'architecte. Le modèle précédent était intentionnellement mauvais (trop grand) pour démontrer la régularisation. 

**Votre mission :** Concevez l'architecture optimale pour ces 15 variables de Kepler. Vous pouvez réduire la profondeur, utiliser `BatchNormalization`, `Dropout`, ou ajuster le `batch_size`. L'objectif est d'obtenir le modèle le plus performant et stable sur le jeu de Test final.

In [ ]:
# 1. Définition de VOTRE architecture
model_final = Sequential([
    # ... À vous de jouer ...
    
    
    
])

# 2. Compilation


# 3. Définition de vos Callbacks si nécessaire


# 4. Entraînement


# 5. Affichage des courbes d'apprentissage



Évaluons enfin votre chef-d'œuvre sur le sous-ensemble de Test que le réseau n'a jamais vu.

In [ ]:
test_loss, test_acc = model_final.evaluate(..., ...)
print(f"\nPerformance sur le jeu de test inédit : {test_acc:.4f}")

**Analyse finale :** Double-cliquez ici pour justifier vos choix d'architecture (profondeur, largeur), vos méthodes de régularisation, et commenter votre résultat final sur le jeu de test par rapport au modèle de base initial.